In [ ]:
import pandas as pd 
import torch
from plotnine import *
import numpy as np

from vpop_calibration import *

%load_ext autoreload
%autoreload 2

In [ ]:
df = pd.read_csv("Mavoglurant_Benchmark_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
      .rename(
          columns={
              "ID": "id",
              "TIME": "time",
              "DV": "value",
              "DOSE": "dose"

          }
      )
      [["id", "time", "value", "dose", "WT"]]
      .astype({"value": "float"})
)
obs_df["output_name"] = "C15"
obs_df["protocol_arm"] = "identity"
display(df.head())
display(obs_df.head())

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    path_to_solving_options="sv.json",
    inputs=["KbBR", "CLint","KbMU","KbAD","KbBO","KbRB","dose","WT"],
    outputs=["C15"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

In [ ]:
prior = {
    "model_intrinsic": {
        "KbBR": {"prior": np.exp(1.1)},
        "KbMU": {"prior": np.exp(0.3)},                
        "KbBO": {"prior": np.exp(0.03)},
        "KbAD": {"prior": np.exp(2)},
        "KbRB": {"prior": np.exp(0.3)}  
    },
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5},
    },
    "pdk": {
        "WT",
        "dose"
    },
    "error_model": {
        "C15": {"error_type": "additive", "sigma": 0.5},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_phase1_iterations=10,
        nb_phase2_iterations=10,
        plot_frames=5,
        optim_max_fun=2,
        verbose=False,
        mcmc_first_burn_in=0,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
)

nlme_model = NlmeModel(
    df=obs_df, prior_params=prior, structural_model=struct_model, config=config
)

In [ ]:
nlme_model.optimizer.run()